# `programming-language-identification-100plus-lite` — PyTorch

2.35M-param byte-level classifier across 107 programming languages. No tokenizer; raw UTF-8 bytes padded to 1023.

Self-contained: this notebook inlines the model definition (vendored from PleIAs/CommonLingua, Apache-2.0) and downloads the checkpoint from the Hub. Run end-to-end in Colab or Jupyter.

## Install dependencies

In [ ]:
%%capture
!pip install -q -U torch huggingface_hub


## Model definition (ByteHybrid — vendored from PleIAs/CommonLingua, Apache-2.0)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ByteNgramEmbed(nn.Module):
    def __init__(self, num_buckets=4096, embed_dim=64, n=3):
        super().__init__()
        self.n, self.num_buckets = n, num_buckets
        self.embed = nn.Embedding(num_buckets, embed_dim)

    def forward(self, byte_ids):
        B, T = byte_ids.shape
        clamped = byte_ids.clamp(max=255)
        padded = F.pad(clamped, (0, self.n - 1), value=0)
        h = torch.zeros(B, T, dtype=torch.long, device=byte_ids.device)
        for i in range(self.n):
            h = h * 257 + padded[:, i:i + T]
        return self.embed(h % self.num_buckets)


class ByteConvBlock(nn.Module):
    def __init__(self, d_model, kernel_size=15, expand=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.pad = kernel_size - 1
        self.conv = nn.Conv1d(d_model, d_model, kernel_size, groups=d_model)
        self.norm2 = nn.LayerNorm(d_model)
        ffn = d_model * expand
        self.ffn_gate = nn.Linear(d_model, ffn, bias=False)
        self.ffn_up = nn.Linear(d_model, ffn, bias=False)
        self.ffn_down = nn.Linear(ffn, d_model, bias=False)

    def forward(self, x):
        residual = x
        x = self.norm1(x).transpose(1, 2)
        x = F.pad(x, (self.pad, 0))
        x = F.silu(self.conv(x)).transpose(1, 2)
        x = residual + x
        residual = x
        x = self.norm2(x)
        return residual + self.ffn_down(F.silu(self.ffn_gate(x)) * self.ffn_up(x))


def _rope(q, k):
    head_dim, seq_len = q.shape[-1], q.shape[-2]
    freqs = 1.0 / (10000.0 ** (torch.arange(0, head_dim, 2, device=q.device).float() / head_dim))
    a = torch.outer(torch.arange(seq_len, device=q.device), freqs)
    cos, sin = a.cos().to(q.dtype), a.sin().to(q.dtype)
    def rot(x):
        x1, x2 = x[..., : head_dim // 2], x[..., head_dim // 2:]
        return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)
    return rot(q), rot(k)


class ByteAttnBlock(nn.Module):
    def __init__(self, d_model, n_heads=4, expand=2):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.norm1 = nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.norm2 = nn.LayerNorm(d_model)
        ffn = d_model * expand
        self.ffn_gate = nn.Linear(d_model, ffn, bias=False)
        self.ffn_up = nn.Linear(d_model, ffn, bias=False)
        self.ffn_down = nn.Linear(ffn, d_model, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        residual = x
        h = self.norm1(x)
        qkv = self.qkv(h).reshape(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = (t.transpose(1, 2) for t in qkv.unbind(dim=2))
        q, k = _rope(q, k)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=None, is_causal=False)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        x = residual + self.out_proj(out)
        residual = x
        h = self.norm2(x)
        return residual + self.ffn_down(F.silu(self.ffn_gate(h)) * self.ffn_up(h))


class ByteHybrid(nn.Module):
    def __init__(self, num_classes, d_model=256, n_conv=3, n_attn=1, n_heads=4,
                 ffn_expand=2, max_len=512, conv_kernel=15, ngram_buckets=4096, ngram_dim=64):
        super().__init__()
        self.max_len = max_len
        self.embed = nn.Embedding(257, d_model, padding_idx=256)
        self.ngram_embed = ByteNgramEmbed(ngram_buckets, ngram_dim, n=3) if ngram_buckets else None
        if self.ngram_embed is not None:
            self.ngram_proj = nn.Linear(ngram_dim, d_model, bias=False)
        self.conv_layers = nn.ModuleList([ByteConvBlock(d_model, conv_kernel, ffn_expand) for _ in range(n_conv)])
        self.attn_layers = nn.ModuleList([ByteAttnBlock(d_model, n_heads, ffn_expand) for _ in range(n_attn)])
        self.final_norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(0.1), nn.Linear(d_model, num_classes))

    def forward(self, byte_ids):
        pad_mask = byte_ids != 256
        x = self.embed(byte_ids)
        if self.ngram_embed is not None:
            x = x + self.ngram_proj(self.ngram_embed(byte_ids))
        for layer in self.conv_layers:
            x = layer(x)
        for layer in self.attn_layers:
            x = layer(x)
        x = self.final_norm(x)
        mask = pad_mask.unsqueeze(-1).to(x.dtype)
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.head(x)


## Load checkpoint from the Hub

In [ ]:
from huggingface_hub import hf_hub_download
import numpy as np

REPO = 'FrameByFrame/programming-language-identification-100plus-lite'
ckpt_path = hf_hub_download(REPO, 'model.pt')
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

BASE_NGRAM = dict(d_model=256, n_conv=3, n_attn=1, n_heads=4, conv_kernel=15,
                  ngram_buckets=4096, ngram_dim=64)
model = ByteHybrid(num_classes=ckpt['num_classes'], max_len=ckpt['max_len'], **BASE_NGRAM).eval()
model.load_state_dict(ckpt['model_state_dict'])
idx2lang = {v: k for k, v in ckpt['lang2idx'].items()}
MAX_LEN = ckpt['max_len']
print(f'{ckpt["num_classes"]} labels | max_len={MAX_LEN} | params={sum(p.numel() for p in model.parameters())/1e6:.2f}M')

## Helpers

In [ ]:
def encode(texts, max_len=MAX_LEN):
    out = np.full((len(texts), max_len), 256, dtype=np.int64)
    for i, t in enumerate(texts):
        b = t.encode('utf-8', errors='replace')[:max_len]
        out[i, :len(b)] = np.frombuffer(b, dtype=np.uint8)
    return torch.from_numpy(out)


@torch.no_grad()
def predict(texts, top_k=3):
    probs = torch.softmax(model(encode(texts)).float(), dim=-1)
    top_p, top_i = probs.topk(top_k, dim=-1)
    return [[(idx2lang[int(j)], float(p)) for p, j in zip(pr, ix)]
            for pr, ix in zip(top_p.tolist(), top_i.tolist())]

## Predict

In [ ]:
samples = [
    "def fib(n):\n    return n if n < 2 else fib(n-1) + fib(n-2)",
    "fn main() {\n    println!(\"hello, world\");\n}",
    "package main\nimport \"fmt\"\nfunc main() { fmt.Println(\"hi\") }",
    "#include <stdio.h>\nint main() { printf(\"hi\\n\"); return 0; }",
    "SELECT name FROM users WHERE id = 42;",
]
for text, top in zip(samples, predict(samples)):
    print(f'{top[0][0]:<14s}  {top[0][1]:.3f}   ({top[1][0]} {top[1][1]:.2f}, {top[2][0]} {top[2][1]:.2f})  | {text[:60]!r}')